# 🏥 Hybrid Clinical Decision Support System - Colab Training

This notebook trains a multi-class skin disease classifier using:
- **Melanoma** Detection
- **Eczema** Detection
- **Psoriasis** Detection  
- **Acne** Detection
- **Normal/Healthy Skin** Detection

**Updated Features:**
- Uses unified Kaggle dataset: `mateenzahid/skin-diesease` (~819MB)
- Single download instead of 5 separate datasets
- GPU acceleration for faster training (TPU also supported)
- Automatically detects and uses GPU/TPU/CPU
- Handles datasets with mixed folder structures (uses ALL images)

**Hardware Requirements:**
- GPU: Recommended (Runtime → Change runtime type → GPU - T4, V100, or A100)
- TPU: Also supported (Runtime → Change runtime type → TPU)
- RAM: 12.7 GB High-RAM runtime if available
- Disk: ~1GB for dataset and models

## 📦 Step 1: Setup Environment

In [9]:
# Check hardware availability (GPU/TPU)
import torch
import os

print(f"PyTorch version: {torch.__version__}")

# Check for GPU first (prioritized)
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device('cuda')
    use_tpu = False
else:
    # Fall back to TPU if GPU not available
    try:
        import torch_xla
        device = torch_xla.device()
        print(f"✓ TPU detected: {device}")
        print(f"TPU cores available")
        use_tpu = True
    except:
        use_tpu = False
        print("⚠️ No GPU/TPU detected. Training will be slower on CPU.")
        print("Consider: Runtime → Change runtime type → Hardware accelerator → GPU")
        device = torch.device('cpu')

print(f"\nUsing device: {device}")

PyTorch version: 2.10.0+cu128
✓ GPU detected: Tesla T4
GPU Memory: 14.6 GB

Using device: cuda


In [10]:
# Install required packages
!pip install -q timm pyyaml gradio kaggle requests tqdm scikit-learn pillow opencv-python shap

# Install TPU support if using TPU (optional, GPU is prioritized)
try:
    import torch_xla
    print("✓ TPU libraries already installed")
except:
    print("TPU libraries not installed (GPU will be used if available)")
    # Uncomment below if you want to install TPU support:
    # !pip install -q cloud-tpu-client==0.10 torch-xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

TPU libraries not installed (GPU will be used if available)


## 💾 Step 2: Mount Google Drive (Optional - for saving models)

In [11]:
# Mount Google Drive to save trained models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
project_dir = '/content/drive/MyDrive/clinical_diagnosis_system'
os.makedirs(project_dir, exist_ok=True)
print(f"✓ Project directory: {project_dir}")

Mounted at /content/drive
✓ Project directory: /content/drive/MyDrive/clinical_diagnosis_system


## 📥 Step 3: Clone Project from GitHub (or upload files)

In [12]:
# Option A: Clone from GitHub (if you have a repo)
# !git clone https://github.com/yourusername/your-repo.git
# %cd your-repo

# Option B: Upload project files manually
# Use Colab's file upload feature to upload your project zip

# Option C: Start fresh (download individual files)
# This notebook assumes you have the project structure

# For this example, we'll create minimal structure
import os
os.makedirs('src', exist_ok=True)
os.makedirs('config', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("✓ Directory structure created")

✓ Directory structure created


## 🔑 Step 4: Setup Kaggle API Credentials

In [13]:
# Upload your kaggle.json file
# Download from: https://www.kaggle.com/settings → API → Create New API Token

from google.colab import files
import os

print("📤 Upload your kaggle.json file when prompted...")
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle API configured!")

📤 Upload your kaggle.json file when prompted...


Saving kaggle.json to kaggle (1).json
✓ Kaggle API configured!


## 📊 Step 5: Download Datasets

In [14]:
# Download unified skin disease dataset from Kaggle
import kaggle
from pathlib import Path

data_dir = Path('data')
raw_dir = data_dir / 'skin_lesions_raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Use unified dataset (819MB total with all 5 disease classes including normal skin)
DATASET_ID = 'mateenzahid/skin-diesease'

print("📥 Downloading unified skin disease dataset...")
print(f"Dataset: {DATASET_ID}")
print("This contains melanoma, eczema, psoriasis, acne, and normal skin images")
print("Total size: ~819MB\n")

try:
    kaggle.api.dataset_download_files(
        DATASET_ID,
        path=str(raw_dir),
        unzip=True,
        quiet=False
    )
    print("\n✓ Dataset downloaded successfully!")

    # List downloaded structure
    print("\n📁 Downloaded structure:")
    for item in sorted(raw_dir.iterdir()):
        if item.is_dir():
            img_count = len(list(item.rglob('*.jpg'))) + len(list(item.rglob('*.png')))
            print(f"  - {item.name}/  ({img_count} images)")
except Exception as e:
    print(f"✗ Error downloading dataset: {e}")
    print("Please check your Kaggle API credentials and dataset permissions")

📥 Downloading unified skin disease dataset...
Dataset: mateenzahid/skin-diesease
This contains melanoma, eczema, psoriasis, acne, and normal skin images
Total size: ~819MB

Dataset URL: https://www.kaggle.com/datasets/mateenzahid/skin-diesease


100%|██████████| 748M/748M [00:04<00:00, 168MB/s]




✓ Dataset downloaded successfully!

📁 Downloaded structure:
  - skin_lesions_raw/  (24298 images)


## 🔧 Step 6: Verify Dataset Structure

The unified dataset already contains organized train/val/test splits, so we'll use it directly without reorganization.

In [15]:
# Verify dataset structure and use existing train/val/test splits
from pathlib import Path

def verify_dataset_structure():
    """Verify the downloaded dataset structure and locate train/val/test splits"""
    raw_dir = Path('data/skin_lesions_raw/skin_lesions_raw')

    disease_mapping = {
        'melanoma': 'Melanoma Skin Cancer Nevi and Moles',
        'eczema': 'Eczema Photos',
        'psoriasis': 'Psoriasis pictures Lichen Planus and related diseases',
        'acne': 'Acne and Rosacea Photos',
        'normal': 'Normal Healthy Skin'
    }

    print("📁 Verifying dataset structure...\n")

    for short_name, full_name in disease_mapping.items():
        disease_dir = raw_dir / short_name

        if not disease_dir.exists():
            print(f"⚠️  {short_name} directory not found")
            continue

        print(f"✓ {short_name}:")

        # Check for existing train/val/test structure
        has_splits = False
        for split in ['train', 'val', 'test', 'valid']:
            split_dir = disease_dir / split
            if split_dir.exists() and split_dir.is_dir():
                img_count = len(list(split_dir.rglob('*.jpg'))) + len(list(split_dir.rglob('*.png')))
                if img_count > 0:
                    print(f"  - {split}/: {img_count} images")
                    has_splits = True

        # If no splits found, count all images
        if not has_splits:
            img_count = len(list(disease_dir.rglob('*.jpg'))) + len(list(disease_dir.rglob('*.png')))
            print(f"  - Total images: {img_count}")

    print("\n✓ Dataset structure verified!")
    print("We'll use the existing folder structure directly (no copying/reorganization needed)")

verify_dataset_structure()

📁 Verifying dataset structure...

✓ melanoma:
  - Total images: 10605
✓ eczema:
  - Total images: 3123
✓ psoriasis:
  - Total images: 2801
✓ acne:
  - train/: 2778 images
  - test/: 918 images
  - valid/: 921 images
✓ normal:
  - Total images: 3152

✓ Dataset structure verified!
We'll use the existing folder structure directly (no copying/reorganization needed)


## 🎓 Step 7: Train the CNN Model

In [16]:
# If you have the full project uploaded, use:
# !python train.py --train-cnn

# Otherwise, use this standalone training code:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Prepare data from existing dataset structure
def prepare_data():
    """Load data from the downloaded dataset (uses existing structure)"""
    from sklearn.model_selection import train_test_split

    raw_dir = Path('data/skin_lesions_raw/skin_lesions_raw')

    class_names = [
        'Melanoma Skin Cancer Nevi and Moles',
        'Eczema Photos',
        'Psoriasis pictures Lichen Planus and related diseases',
        'Acne and Rosacea Photos',
        'Normal Healthy Skin'
    ]

    # Map short names to full class names
    disease_mapping = {
        'melanoma': 'Melanoma Skin Cancer Nevi and Moles',
        'eczema': 'Eczema Photos',
        'psoriasis': 'Psoriasis pictures Lichen Planus and related diseases',
        'acne': 'Acne and Rosacea Photos',
        'normal': 'Normal Healthy Skin'
    }

    class_to_idx = {name: i for i, name in enumerate(class_names)}

    train_paths, train_labels = [], []
    val_paths, val_labels = [], []

    print("📊 Loading images from dataset...\n")

    for short_name, full_name in disease_mapping.items():
        disease_dir = raw_dir / short_name

        if not disease_dir.exists():
            print(f"⚠️  Skipping {short_name} - directory not found")
            continue

        # Check for existing train/val/test splits
        train_dir = disease_dir / 'train'
        val_dir = disease_dir / 'val'
        valid_dir = disease_dir / 'valid'  # Some datasets use 'valid' instead of 'val'
        test_dir = disease_dir / 'test'

        # Use val or valid, whichever exists
        if not val_dir.exists() and valid_dir.exists():
            val_dir = valid_dir

        # Check if this disease has pre-split folders
        has_splits = train_dir.exists() and (val_dir.exists() or test_dir.exists())

        if has_splits:
            # Use existing splits
            if train_dir.exists():
                train_count = 0
                for img_path in train_dir.rglob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        train_paths.append(str(img_path))
                        train_labels.append(class_to_idx[full_name])
                        train_count += 1
                print(f"✓ {short_name}/train: {train_count} images")

            if val_dir.exists():
                val_count = 0
                for img_path in val_dir.rglob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        val_paths.append(str(img_path))
                        val_labels.append(class_to_idx[full_name])
                        val_count += 1
                print(f"✓ {short_name}/val: {val_count} images")
            elif test_dir.exists():
                val_count = 0
                for img_path in test_dir.rglob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        val_paths.append(str(img_path))
                        val_labels.append(class_to_idx[full_name])
                        val_count += 1
                print(f"✓ {short_name}/test (used as val): {val_count} images")
        else:
            # No splits found - collect all images and split them manually
            all_paths = []
            for img_path in disease_dir.rglob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    all_paths.append(str(img_path))

            if len(all_paths) > 0:
                # Split 80% train, 20% val
                all_labels = [class_to_idx[full_name]] * len(all_paths)
                disease_train, disease_val, labels_train, labels_val = train_test_split(
                    all_paths, all_labels, test_size=0.2, random_state=42, stratify=None
                )

                train_paths.extend(disease_train)
                train_labels.extend(labels_train)
                val_paths.extend(disease_val)
                val_labels.extend(labels_val)

                print(f"✓ {short_name}: {len(all_paths)} images total → {len(disease_train)} train, {len(disease_val)} val")
            else:
                print(f"⚠️  {short_name}: No images found")

    print(f"\n📈 Dataset Summary:")
    print(f"Training samples: {len(train_paths)}")
    print(f"Validation samples: {len(val_paths)}")
    print(f"Total images: {len(train_paths) + len(val_paths)}")

    return train_paths, train_labels, val_paths, val_labels, class_names

train_paths, train_labels, val_paths, val_labels, class_names = prepare_data()

📊 Loading images from dataset...

✓ melanoma: 10605 images total → 8484 train, 2121 val
✓ eczema: 3123 images total → 2498 train, 625 val
✓ psoriasis: 2806 images total → 2244 train, 562 val
✓ acne/train: 2778 images
✓ acne/val: 921 images
✓ normal: 3152 images total → 2521 train, 631 val

📈 Dataset Summary:
Training samples: 18525
Validation samples: 4860
Total images: 23385


In [18]:
# Define transforms - using larger image size for better quality
# Note: Larger images require more GPU memory. Reduce batch_size if you get OOM errors.
IMG_SIZE = 384  # Increased from 224 for better detail preservation

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets and dataloaders
train_dataset = SkinLesionDataset(train_paths, train_labels, train_transform)
val_dataset = SkinLesionDataset(val_paths, val_labels, val_transform)

# Reduced batch size for larger images (adjust based on GPU memory)
batch_size = 16  # Reduced from 32 due to larger image size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {batch_size}")
print(f"Batches per epoch: {len(train_loader)}")

Image size: 384x384
Batch size: 16
Batches per epoch: 1158


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    use_tpu = False
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_xla
        device = torch_xla.device()
        use_tpu = True
        print(f"✓ Using TPU: {device}")
    except:
        device = torch.device('cpu')
        use_tpu = False
        print(f"⚠️ Using CPU (slower)")

# Model
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)
model = model.to(device)

# Loss
criterion = nn.CrossEntropyLoss()

# Debug
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Model initialized")

✓ Using GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model parameters: 4,013,953
✓ Model initialized


In [ ]:
# Training loop with TPU/GPU support
def train_model(model, train_loader, val_loader, epochs=10):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # Check if using TPU
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        use_tpu = True
    except:
        use_tpu = False

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            if use_tpu:
                xm.optimizer_step(optimizer)  # TPU-specific optimizer step
            else:
                optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': train_loss/len(pbar), 'acc': 100.*train_correct/train_total})

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total

        print(f"\nEpoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            # For TPU, move model to CPU before saving
            if use_tpu:
                cpu_model = model.cpu()
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': cpu_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')
                model = model.to(device)  # Move back to TPU
            else:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')

            print(f"  ✓ Best model saved (Val Acc: {val_acc:.2f}%)")

    return history

# Train for 10 epochs (increase for better results)
history = train_model(model, train_loader, val_loader, epochs=10)

Epoch 1/10: 100%|██████████| 1158/1158 [05:10<00:00,  3.73it/s, loss=0.293, acc=88.1]



Epoch 1:
  Train Loss: 0.2925, Train Acc: 88.15%
  Val Loss: 0.1654, Val Acc: 92.35%
  ✓ Best model saved (Val Acc: 92.35%)


Epoch 2/10: 100%|██████████| 1158/1158 [05:11<00:00,  3.72it/s, loss=0.29, acc=87.8]



Epoch 2:
  Train Loss: 0.2900, Train Acc: 87.82%
  Val Loss: 0.1471, Val Acc: 91.58%


Epoch 3/10:  33%|███▎      | 387/1158 [01:44<03:10,  4.04it/s, loss=0.0637, acc=91.2]

## 📊 Step 8: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n🎯 Final Results:")
print(f"  Best Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"  Final Training Accuracy: {history['train_acc'][-1]:.2f}%")

## 💾 Step 9: Save Model to Google Drive

In [ ]:
# Copy trained model to Google Drive
!cp models/cnn_skin_lesion.pth /content/drive/MyDrive/clinical_diagnosis_system/

print("✓ Model saved to Google Drive!")
print("You can now download it and use it locally.")

## 🧪 Step 10: Test the Model

In [ ]:
# Test on a few validation images
from PIL import Image
import random

model.eval()

# Select random validation images
test_indices = random.sample(range(len(val_paths)), 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, test_idx in enumerate(test_indices):
    img_path = val_paths[test_idx]
    true_label = val_labels[test_idx]

    # Load and display image
    img = Image.open(img_path).convert('RGB')
    axes[idx].imshow(img)

    # Predict
    img_tensor = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        pred_label = output.argmax(1).item()
        confidence = probabilities[pred_label].item()

    axes[idx].set_title(f"True: {class_names[true_label][:15]}...\nPred: {class_names[pred_label][:15]}...\nConf: {confidence:.2%}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## ✅ Summary

You've successfully:
1. ✓ Set up the environment with GPU support (TPU optional)
2. ✓ Downloaded unified skin disease dataset from Kaggle (mateenzahid/skin-diesease)
3. ✓ Loaded ALL images from dataset (~24,000+ images) with smart split handling
4. ✓ Trained an EfficientNet-B0 model on 5 disease classes (including Normal skin)
5. ✓ Visualized training results
6. ✓ Saved the model to Google Drive

**Dataset Info:**
- Source: https://www.kaggle.com/datasets/mateenzahid/skin-diesease
- Size: ~819MB total (~24,000+ images)
- Classes: Melanoma, Eczema, Psoriasis, Acne, Normal/Healthy Skin
- Structure: Uses ALL images - handles both split and non-split folder structures
  - Diseases with splits (acne): Uses existing train/val/test folders
  - Diseases without splits (melanoma, eczema, psoriasis, normal): Auto-splits 80/20
- All original data preserved in data/skin_lesions_raw/

**Model Performance:**
- The model now correctly identifies normal/healthy skin to avoid false positives
- Uses GPU acceleration for faster training (TPU also supported)
- EfficientNet-B0 backbone with transfer learning
- Trains on ~20,000 images (80% of total dataset)

### Next Steps:
- Download the model from Google Drive (`cnn_skin_lesion.pth`)
- Place it in your local project: `models/cnn_skin_lesion.pth`
- Run the web interface: `python app.py`
- Test with both diseased and healthy skin images

### Tips for Better Results:
- Train for 20-50 epochs (this notebook uses 10 for speed)
- Use learning rate scheduling
- Add test-time augmentation
- Try ensemble methods with multiple models
- Increase batch size if using GPU with sufficient memory (32-64 works well on T4/V100)